# Laboratory Task 6 CNN Implementation

**Name:** Ara Galario  
**4th Year DS4A**

**Instruction:** Convert the given CNN architecture diagram into a PyTorch CNN.

This solution follows the diagram, including its kernel sizes, strides, padding, dropout, and fully connected layers.

## Architecture from the Diagram

Input: **1 × 28 × 28**, batch size 32

- Conv1: 1 → 32, kernel 3×3, stride 1, padding 1, then ReLU
- MaxPool1: kernel 2×2, stride 1, padding 1
- Conv2: 32 → 64, kernel 3×3, stride 1, padding 1, then ReLU
- Conv3: 64 → 128, kernel 3×3, stride 1, padding 1, then ReLU
- Conv4: 128 → 256, kernel 3×3, stride 1, padding 1, then ReLU
- MaxPool2: kernel 2×2, stride 1, padding 0
- Dropout: p = 0.2
- Flatten
- FCN1: flattened features → 1000, then ReLU
- FCN2: 1000 → 500, then ReLU
- FCN3: 500 → 10, then Softmax

The activity uses **10 output classes**, so FCN3 has 10 outputs.

## 1. Import Libraries

In [1]:
import torch
import torch.nn as nn

## 2. Work Out the Feature-Map Sizes

Output size = floor((Input + 2P − K) / S) + 1, where P is padding, K is kernel size, and S is stride.

In [2]:
def out_size(size, k, s, p):
    return (size + 2 * p - k) // s + 1

s = 28
s = out_size(s, 3, 1, 1); print("After Conv1   :", s)
s = out_size(s, 2, 1, 1); print("After MaxPool1:", s)
s = out_size(s, 3, 1, 1); print("After Conv2   :", s)
s = out_size(s, 3, 1, 1); print("After Conv3   :", s)
s = out_size(s, 3, 1, 1); print("After Conv4   :", s)
s = out_size(s, 2, 1, 0); print("After MaxPool2:", s)

flat = 256 * s * s
print("Flatten size  :", flat)

After Conv1   : 28
After MaxPool1: 29
After Conv2   : 29
After Conv3   : 29
After Conv4   : 29
After MaxPool2: 28
Flatten size  : 200704


## 3. Build the CNN

In [3]:
model = nn.Sequential(
    nn.Conv2d(1, 32, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=1, padding=1),

    nn.Conv2d(32, 64, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.Conv2d(64, 128, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.Conv2d(128, 256, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=1, padding=0),

    nn.Dropout(0.2),
    nn.Flatten(),

    nn.Linear(flat, 1000),
    nn.ReLU(),
    nn.Linear(1000, 500),
    nn.ReLU(),
    nn.Linear(500, 10),
    nn.Softmax(dim=1)
)

print(model)

Sequential(
  (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=1, padding=1, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU()
  (7): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU()
  (9): MaxPool2d(kernel_size=2, stride=1, padding=0, dilation=1, ceil_mode=False)
  (10): Dropout(p=0.2, inplace=False)
  (11): Flatten(start_dim=1, end_dim=-1)
  (12): Linear(in_features=200704, out_features=1000, bias=True)
  (13): ReLU()
  (14): Linear(in_features=1000, out_features=500, bias=True)
  (15): ReLU()
  (16): Linear(in_features=500, out_features=10, bias=True)
  (17): Softmax(dim=1)
)


## 4. Check the Shapes with a Dummy Batch

A random batch of shape `(32, 1, 28, 28)` is passed through the model one layer at a time.

In [4]:
model.eval()
x = torch.randn(32, 1, 28, 28)
print("Input:", tuple(x.shape))

with torch.no_grad():
    for layer in model:
        x = layer(x)
        print(f"{layer.__class__.__name__:10s}", tuple(x.shape))

Input: (32, 1, 28, 28)
Conv2d     (32, 32, 28, 28)
ReLU       (32, 32, 28, 28)
MaxPool2d  (32, 32, 29, 29)
Conv2d     (32, 64, 29, 29)
ReLU       (32, 64, 29, 29)
Conv2d     (32, 128, 29, 29)
ReLU       (32, 128, 29, 29)
Conv2d     (32, 256, 29, 29)
ReLU       (32, 256, 29, 29)
MaxPool2d  (32, 256, 28, 28)
Dropout    (32, 256, 28, 28)
Flatten    (32, 200704)
Linear     (32, 1000)
ReLU       (32, 1000)
Linear     (32, 500)
ReLU       (32, 500)
Linear     (32, 10)
Softmax    (32, 10)


## Expected Shapes

| Layer | Output (batch of 32) |
|---|---|
| Input | (32, 1, 28, 28) |
| Conv1 | (32, 32, 28, 28) |
| MaxPool1 | (32, 32, 29, 29) |
| Conv2 | (32, 64, 29, 29) |
| Conv3 | (32, 128, 29, 29) |
| Conv4 | (32, 256, 29, 29) |
| MaxPool2 | (32, 256, 28, 28) |
| Dropout | (32, 256, 28, 28) |
| Flatten | (32, 200704) |
| FCN1 | (32, 1000) |
| FCN2 | (32, 500) |
| FCN3 + Softmax | (32, 10) |

**Note:** MaxPool1 gives 29 × 29 (not 14 × 14) because the diagram specifies stride 1 and padding 1.